# Titanic Survival Analysis Project

## Project Goal

This project studies the Titanic passenger dataset to understand which factors were associated with survival.

The main question is:

**Which passenger characteristics helped explain whether a passenger survived or not?**

We will use a beginner-friendly workflow:

1. Load and inspect the dataset  
2. Clean missing values  
3. Explore survival patterns visually and numerically  
4. Prepare the data for machine learning  
5. Build classification models  
6. Evaluate model performance  
7. Interpret the most important survival predictors  

Although this is sometimes called a "survival analysis" project, the Titanic dataset usually does **not** contain time-to-event information. So here, survival means a binary outcome:

- `1` = survived  
- `0` = did not survive

## 1. Import Libraries

We begin by importing the Python libraries needed for data analysis, visualization, and machine learning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, RocCurveDisplay

import warnings
warnings.filterwarnings("ignore")

## 2. Load the Titanic Dataset

This notebook first tries to load the Titanic dataset from `seaborn`. If that does not work, it uses a public GitHub CSV version of the same commonly used dataset.

The dataset contains passenger-level information such as class, sex, age, fare, number of relatives aboard, and survival status.

In [ ]:
try:
    import seaborn as sns
    titanic = sns.load_dataset("titanic")
    print("Dataset loaded from seaborn.")
except Exception:
    url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
    titanic = pd.read_csv(url)
    print("Dataset loaded from GitHub URL.")

titanic.head()

## 3. Basic Dataset Inspection

Before modeling, we need to understand the structure of the data.

Important questions:

- How many rows and columns are there?
- Which columns are numerical?
- Which columns are categorical?
- Are there missing values?

In [ ]:
print("Shape of the dataset:", titanic.shape)
titanic.info()

In [ ]:
titanic.describe(include="all").T

In [ ]:
missing_values = titanic.isnull().sum().sort_values(ascending=False)
missing_values[missing_values > 0]

## 4. Understanding the Target Variable

The target variable is `survived`.

- `1` means the passenger survived
- `0` means the passenger did not survive

We first examine the overall survival rate.

In [ ]:
survival_counts = titanic["survived"].value_counts()
survival_rate = titanic["survived"].mean()

print(survival_counts)
print(f"Overall survival rate: {survival_rate:.2%}")

In [ ]:
survival_counts.plot(kind="bar")
plt.title("Survival Counts")
plt.xlabel("Survived")
plt.ylabel("Number of Passengers")
plt.xticks(rotation=0)
plt.show()

## 5. Exploratory Data Analysis

Now we examine how survival differed by major passenger characteristics.

We will look at:

- Sex
- Passenger class
- Age
- Fare
- Family size
- Embarkation location

### Survival by Sex

Historically, women and children were often prioritized during evacuation. We can test whether survival rates were higher for female passengers.

In [ ]:
survival_by_sex = titanic.groupby("sex")["survived"].mean().sort_values(ascending=False)
survival_by_sex

In [ ]:
survival_by_sex.plot(kind="bar")
plt.title("Survival Rate by Sex")
plt.xlabel("Sex")
plt.ylabel("Survival Rate")
plt.xticks(rotation=0)
plt.show()

### Survival by Passenger Class

Passenger class may reflect wealth, cabin location, and access to lifeboats. We compare survival rates across first, second, and third class.

In [ ]:
survival_by_class = titanic.groupby("pclass")["survived"].mean()
survival_by_class

In [ ]:
survival_by_class.plot(kind="bar")
plt.title("Survival Rate by Passenger Class")
plt.xlabel("Passenger Class")
plt.ylabel("Survival Rate")
plt.xticks(rotation=0)
plt.show()

### Survival by Age

Age may matter because children may have had better survival chances. We compare age distributions for survivors and non-survivors.

In [ ]:
titanic.boxplot(column="age", by="survived")
plt.title("Age Distribution by Survival Status")
plt.suptitle("")
plt.xlabel("Survived")
plt.ylabel("Age")
plt.show()

In [ ]:
titanic.groupby("survived")["age"].describe()

### Survival by Fare

Fare can act as a proxy for wealth, class, and cabin location. Higher fares may be associated with higher survival rates.

In [ ]:
titanic.boxplot(column="fare", by="survived")
plt.title("Fare Distribution by Survival Status")
plt.suptitle("")
plt.xlabel("Survived")
plt.ylabel("Fare")
plt.show()

In [ ]:
titanic.groupby("survived")["fare"].describe()

## 6. Feature Engineering

Feature engineering means creating useful new variables from existing data.

Here we create:

- `family_size`: total number of family members aboard plus the passenger
- `is_alone`: whether the passenger traveled alone
- `age_group`: simple age categories

These variables may help explain survival.

In [ ]:
df = titanic.copy()

df["family_size"] = df["sibsp"] + df["parch"] + 1
df["is_alone"] = (df["family_size"] == 1).astype(int)

df["age_group"] = pd.cut(
    df["age"],
    bins=[0, 12, 18, 35, 60, 100],
    labels=["child", "teen", "young_adult", "adult", "senior"]
)

df[["sibsp", "parch", "family_size", "is_alone", "age", "age_group"]].head()

### Survival by Family Size

This checks whether passengers traveling alone or with family had different survival rates.

In [ ]:
family_survival = df.groupby("family_size")["survived"].mean()
family_survival

In [ ]:
family_survival.plot(kind="bar")
plt.title("Survival Rate by Family Size")
plt.xlabel("Family Size")
plt.ylabel("Survival Rate")
plt.xticks(rotation=0)
plt.show()

### Survival by Age Group

Instead of using exact age only, age groups can make the pattern easier to understand.

In [ ]:
age_group_survival = df.groupby("age_group")["survived"].mean()
age_group_survival

In [ ]:
age_group_survival.plot(kind="bar")
plt.title("Survival Rate by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Survival Rate")
plt.xticks(rotation=45)
plt.show()

## 7. Select Features for Modeling

We now choose a set of predictors.

The target is:

- `survived`

The predictors include:

- Passenger class
- Sex
- Age
- Fare
- Family variables
- Embarkation location
- Alone status
- Age group

We avoid using columns such as `alive`, because it directly repeats the survival outcome and would cause data leakage.

In [ ]:
features = [
    "pclass", "sex", "age", "fare", "sibsp", "parch",
    "embarked", "family_size", "is_alone", "age_group"
]

target = "survived"

model_df = df[features + [target]].copy()
model_df.head()

## 8. Train-Test Split

We split the data into training and testing sets.

- The training set is used to fit the model.
- The testing set is used to evaluate how well the model performs on unseen data.

This helps avoid judging the model only on data it has already seen.

In [ ]:
X = model_df[features]
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)

## 9. Preprocessing Pipeline

Machine learning models need numerical input.

We will:

- Fill missing numerical values with the median
- Fill missing categorical values with the most frequent value
- Convert categorical variables into dummy variables using one-hot encoding

Using a pipeline helps keep the workflow organized and prevents mistakes.

In [ ]:
numeric_features = ["pclass", "age", "fare", "sibsp", "parch", "family_size", "is_alone"]
categorical_features = ["sex", "embarked", "age_group"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

## 10. Model 1: Logistic Regression

Logistic regression is a good starting model for binary classification.

It estimates the probability that a passenger survived.

In [ ]:
logistic_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

logistic_model.fit(X_train, y_train)

log_pred = logistic_model.predict(X_test)
log_prob = logistic_model.predict_proba(X_test)[:, 1]

print("Logistic Regression Accuracy:", accuracy_score(y_test, log_pred))
print("Logistic Regression ROC-AUC:", roc_auc_score(y_test, log_prob))
print()
print(classification_report(y_test, log_pred))

In [ ]:
cm = confusion_matrix(y_test, log_pred)
cm_df = pd.DataFrame(cm, index=["Actual 0", "Actual 1"], columns=["Predicted 0", "Predicted 1"])
cm_df

## 11. Model 2: Random Forest

Random forest is a more flexible model. It combines many decision trees and can capture nonlinear relationships and interactions.

For example, the effect of age may depend on sex or passenger class.

In [ ]:
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_depth=5,
        random_state=42
    ))
])

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))
print("Random Forest ROC-AUC:", roc_auc_score(y_test, rf_prob))
print()
print(classification_report(y_test, rf_pred))

In [ ]:
cm = confusion_matrix(y_test, rf_pred)
cm_df = pd.DataFrame(cm, index=["Actual 0", "Actual 1"], columns=["Predicted 0", "Predicted 1"])
cm_df

## 12. Compare Model Performance

Accuracy tells us the share of correct predictions.

ROC-AUC measures how well the model separates survivors from non-survivors across probability thresholds.

A higher ROC-AUC generally means better classification ability.

In [ ]:
model_comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, log_pred),
        accuracy_score(y_test, rf_pred)
    ],
    "ROC_AUC": [
        roc_auc_score(y_test, log_prob),
        roc_auc_score(y_test, rf_prob)
    ]
})

model_comparison

In [ ]:
model_comparison.set_index("Model")[["Accuracy", "ROC_AUC"]].plot(kind="bar")
plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.xticks(rotation=20)
plt.show()

## 13. ROC Curve

The ROC curve helps visualize classification performance.

A model with a curve closer to the top-left corner is generally better.

In [ ]:
RocCurveDisplay.from_predictions(y_test, log_prob, name="Logistic Regression")
RocCurveDisplay.from_predictions(y_test, rf_prob, name="Random Forest")
plt.title("ROC Curves")
plt.show()

## 14. Feature Importance from Random Forest

Random forest can estimate which variables were most important for prediction.

This does not prove causation. It only tells us which features helped the model make predictions.

In [ ]:
# Get feature names after preprocessing
onehot = rf_model.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"]
cat_names = onehot.get_feature_names_out(categorical_features)

all_feature_names = numeric_features + list(cat_names)

importances = rf_model.named_steps["classifier"].feature_importances_

importance_df = pd.DataFrame({
    "Feature": all_feature_names,
    "Importance": importances
}).sort_values("Importance", ascending=False)

importance_df.head(15)

In [ ]:
importance_df.head(15).set_index("Feature")["Importance"].sort_values().plot(kind="barh")
plt.title("Top 15 Random Forest Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

## 15. Main Findings

Based on the exploratory analysis and machine learning models, the most important survival-related factors are usually:

1. **Sex**  
   Female passengers had much higher survival rates.

2. **Passenger class**  
   First-class passengers generally had higher survival rates than third-class passengers.

3. **Fare**  
   Higher fare often reflects higher class or better cabin location, which was associated with higher survival probability.

4. **Age**  
   Children tended to have better survival chances than some adult groups.

5. **Family structure**  
   Traveling alone or with family could affect survival patterns.

The exact model results may vary slightly depending on the train-test split and preprocessing choices.

## 16. Limitations

This project has several limitations:

- The dataset is relatively small.
- Some important variables, such as cabin location, have many missing values.
- Survival was affected by many real-world factors not fully captured in the dataset.
- Machine learning results show association, not direct causation.
- The dataset does not include time-to-event information, so this is not formal survival-time modeling.

Despite these limits, the Titanic dataset is useful for learning classification, data cleaning, and model interpretation.

## 17. Conclusion

This project showed how Python can be used to study survival patterns in the Titanic dataset.

The analysis suggests that survival was strongly associated with sex, passenger class, fare, age, and family structure. Logistic regression provided a simple and interpretable baseline model, while random forest offered a more flexible approach that could capture more complex relationships.

The project also demonstrated a complete beginner-friendly data science workflow:

- Data loading
- Cleaning
- Visualization
- Feature engineering
- Modeling
- Evaluation
- Interpretation

This makes the Titanic dataset a strong introductory project for classification and applied data analysis.

## 18. Reflection Questions

1. Why should we avoid using the `alive` column as a predictor?
2. Why might passenger class be related to survival?
3. Why is train-test splitting important?
4. What does the confusion matrix tell us?
5. Which model would you prefer if interpretability is more important?
6. Which model would you prefer if prediction accuracy is more important?

## 19. Possible Extensions

To make this project stronger, you could add:

- Cross-validation
- Hyperparameter tuning
- Decision tree visualization
- Additional models such as KNN, SVM, or Gradient Boosting
- More detailed cabin analysis
- A short written report explaining the findings
- A dashboard showing survival rates interactively